# Preprocessing of df results
Feature creation of:
- get original image path for each image
- get caotion
- get newssource
- get subject politician
- get surface of politician's face
- get emotion distance
- get iconicity score


In [ ]:
# install libraries
!pip install openpyxl

In [ ]:
# import libraries
import ast
import numpy as np
import pandas as pd
import ast
import numpy as np
import pandas as pd
import json
import re
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

## 0. Feature creation


In [ ]:
path = '/content/drive/MyDrive/Thesis/analysis/df_results.xlsx'
df = pd.read_excel(path)
df = df.drop_duplicates()
df

In [ ]:
# gets original image path for each image
# Checks if it's already in the target format
# Determines original images based on path
def convert_path(path):
    pattern = r'^/content/drive/MyDrive/Thesis/images_dataset/origin/[^/]+/images/\d{4}/\d+\.jpg$'
    if re.match(pattern, path):
        return path
    filename = os.path.basename(path)
    match = re.match(r'(\w+)_images_(\d{4})_(\d+)_\d\.jpg$', filename)

    if not match:
        raise ValueError(f"Filename format not recognized: {filename}")

    category, id1, id2 = match.groups()
    new_path = f"/content/drive/MyDrive/Thesis/images_dataset/origin/{category}/images/{id1}/{id2}.jpg"
    return new_path


df = pd.read_excel(path)
df['original_image'] = df['image_path'].apply(lambda x: 'images_dataset' in x if isinstance(x, str) else False)
df['real_image'] = df['image_path'].apply(convert_path)
df

In [ ]:
politician_files_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/AllPoliticians_captions.xlsx'
df_is_politician_caption = pd.read_excel(politician_files_path)

In [ ]:
# gets caption from data file
# gets newssource from data file
# Removes base path and adds a dot at the beginning
# joins the data together
# data_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/data.json'

# Load politician data
politician_files_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/AllPoliticians_captions.xlsx'
df_is_politician_caption = pd.read_excel(politician_files_path)

data = pd.read_excel(politician_files_path)
df_captions = pd.DataFrame(data)

def modify_path(file_path):
    base_path = '/content/drive/MyDrive/Thesis/images_dataset/origin'
    if file_path.startswith(base_path):
        return '.' + file_path[len(base_path):]
    return file_path

df['real_image_path'] = df['real_image'].apply(modify_path)

df_captions_subset = df_captions[['image_path', 'caption', 'source']].rename(
    columns={'image_path': 'join_key'}
)

df = df.merge(
    df_captions_subset,
    how='left',
    left_on='real_image_path',
    right_on='join_key'
)
df.drop(columns='join_key', inplace=True)

# Load politician data
politician_files_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/AllPoliticians_captions.xlsx'
df_is_politician_caption = pd.read_excel(politician_files_path)

# rename
df_is_politician_caption = df_is_politician_caption.rename(columns={'image_path': 'politician_image_path'})

target_politicians = ['trump', 'merkel', 'rutte', 'obama']
df_is_politician_caption = df_is_politician_caption[df_is_politician_caption['politician'].str.lower().isin(target_politicians)]

# Merge based on image paths
df = df.merge(
    df_is_politician_caption[['politician_image_path', 'politician']],
    how='left',
    left_on='real_image_path',
    right_on='politician_image_path'
)


df['subject_politician'] = df['politician'].str.lower()
df.drop(columns=['politician_image_path', 'politician'], inplace=True)
df

## Calculate surfaces

In [ ]:
# Constants
IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024

# calculate surface area from bounding box
def calculate_surface(s):
    if pd.isna(s):
        return np.nan
    try:
        x1, y1, x2, y2 = ast.literal_eval(s)
        pixel_x1 = x1 * IMAGE_WIDTH
        pixel_y1 = y1 * IMAGE_HEIGHT
        pixel_x2 = x2 * IMAGE_WIDTH
        pixel_y2 = y2 * IMAGE_HEIGHT
        width = pixel_x2 - pixel_x1
        height = pixel_y2 - pixel_y1
        return width * height
    except (ValueError, SyntaxError, TypeError):
        return np.nan

# get politicians
politicians = ['donald_trump', 'angela_merkel', 'mark_rutte', 'barack_obama']

# Apply the function for each politician and store result in new column
for name in politicians:
    col_name = f'box_face_{name}'
    surface_col_name = f'surface_of_{name}_box'
    df[surface_col_name] = df[col_name].apply(calculate_surface)


## Emotion similarity matrix

In [ ]:
# showing distance between emotions
emotion_similarity = {
    'angry':     {'angry': 1.00, 'disgust': 0.85, 'fear': 0.70, 'happy': 0.10, 'neutral': 0.40, 'sad': 0.75, 'surprise': 0.20},
    'disgust':   {'angry': 0.85, 'disgust': 1.00, 'fear': 0.65, 'happy': 0.05, 'neutral': 0.35, 'sad': 0.70, 'surprise': 0.15},
    'fear':      {'angry': 0.70, 'disgust': 0.65, 'fear': 1.00, 'happy': 0.10, 'neutral': 0.45, 'sad': 0.60, 'surprise': 0.30},
    'happy':     {'angry': 0.10, 'disgust': 0.05, 'fear': 0.10, 'happy': 1.00, 'neutral': 0.70, 'sad': 0.10, 'surprise': 0.75},
    'neutral':   {'angry': 0.40, 'disgust': 0.35, 'fear': 0.45, 'happy': 0.70, 'neutral': 1.00, 'sad': 0.50, 'surprise': 0.60},
    'sad':       {'angry': 0.75, 'disgust': 0.70, 'fear': 0.60, 'happy': 0.10, 'neutral': 0.50, 'sad': 1.00, 'surprise': 0.25},
    'surprise':  {'angry': 0.20, 'disgust': 0.15, 'fear': 0.30, 'happy': 0.75, 'neutral': 0.60, 'sad': 0.25, 'surprise': 1.00},
}


politician_column_map = {
    'trump': 'donald_trump',
    'merkel': 'angela_merkel',
    'rutte': 'mark_rutte',
    'obama': 'barack_obama'
}

# filter original and generated images
df_original_images2 = df[df['original_image'] == True].copy()
df_generated_images2 = df[df['original_image'] == False].copy()

# Create emotion profile per politician
emotion_distributions = {}
for politician_key, emotion_suffix in politician_column_map.items():
    emotion_col = f'emotion_face_{emotion_suffix}'
    subset = df_original_images2[df_original_images2['subject_politician'] == politician_key]
    emotion_counts = subset[emotion_col].value_counts(normalize=True).to_dict()
    emotion_distributions[politician_key] = emotion_counts

# create distance calculator
def compute_similarity_to_profile(row, column_map, distribution_map):
    politician_key = row['subject_politician']
    emotion_col = f"emotion_face_{column_map.get(politician_key)}"
    emotion = row.get(emotion_col)
    if pd.isna(emotion):
        return None

    profile = distribution_map.get(politician_key, {})
    score = 0.0
    for ref_emotion, prob in profile.items():
        sim = emotion_similarity.get(emotion, {}).get(ref_emotion, 0.0)
        score += prob * sim
    return score

# calculate for all rows in both original and generated images
df_original_images2['emotion_similarity_to_profile'] = df_original_images2.apply(
    lambda row: compute_similarity_to_profile(row, politician_column_map, emotion_distributions), axis=1
)

df_generated_images2['emotion_similarity_to_profile'] = df_generated_images2.apply(
    lambda row: compute_similarity_to_profile(row, politician_column_map, emotion_distributions), axis=1
)

In [ ]:
# concat the two df's into one
df = pd.concat([df_original_images2, df_generated_images2], ignore_index=True)

## Calculate Iconicity score

In [ ]:
import pandas as pd
import numpy as np
import ast

# Constants
IMG_WIDTH, IMG_HEIGHT = 1024, 1024
IMG_CENTER_X, IMG_CENTER_Y = IMG_WIDTH / 2, IMG_HEIGHT / 2
MAX_DIST = np.sqrt(IMG_CENTER_X**2 + IMG_CENTER_Y**2)

# Emotion sentiment mapping
emotion_map = {
    'happy': 1.0,
    'surprise': 0.8,
    'neutral': 0.5,
    'sad': 0.2,
    'angry': 0.1,
    'fear': 0.1,
    'disgust': 0.1
}

# Shortname to full column mapping
politician_column_map = {
    'trump': 'donald_trump',
    'obama': 'barack_obama',
    'merkel': 'angela_merkel',
    'rutte': 'mark_rutte'
}

# Normalize helper
def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-8)

# Main function
def compute_iconicity(df):
    results = []

    for _, row in df.iterrows():
        shorthand = row.get('subject_politician')
        if not isinstance(shorthand, str):
            continue

        shorthand = shorthand.strip().lower()
        p = politician_column_map.get(shorthand)
        if not p:
            continue

        # centering score
        face_center = row.get(f'center_face_{p}', [np.nan, np.nan])
        if isinstance(face_center, str):
            try:
                face_center = ast.literal_eval(face_center)
            except:
                face_center = [np.nan, np.nan]
        if isinstance(face_center, list) and len(face_center) == 2:
            face_x = face_center[0] * IMG_WIDTH
            face_y = face_center[1] * IMG_HEIGHT
        else:
            face_x, face_y = np.nan, np.nan
        face_distance = np.sqrt((face_x - IMG_CENTER_X)**2 + (face_y - IMG_CENTER_Y)**2)
        C_raw = 1 - face_distance / MAX_DIST

        # surface sciore size
        face_box = row.get(f'box_face_{p}', [0, 0, 0, 0])
        if isinstance(face_box, str):
            try:
                face_box = ast.literal_eval(face_box)
            except:
                face_box = [0, 0, 0, 0]
        if isinstance(face_box, list) and len(face_box) == 4:
            x_min, y_min, x_max, y_max = face_box
            face_w = (x_max - x_min) * IMG_WIDTH
            face_h = (y_max - y_min) * IMG_HEIGHT
        else:
            face_w, face_h = 0, 0
        S_raw = (face_w * face_h) / (IMG_WIDTH * IMG_HEIGHT)

        # visibility
        upper_parts = [
            f'NOSE_{p}', f'LEFT_SHOULDER_{p}', f'RIGHT_SHOULDER_{p}',
            f'LEFT_ELBOW_{p}', f'RIGHT_ELBOW_{p}',
            f'LEFT_WRIST_{p}', f'RIGHT_WRIST_{p}'
        ]
        visible_parts = 0
        for part in upper_parts:
            val = row.get(part)
            if isinstance(val, str):
                try:
                    val = ast.literal_eval(val)
                except:
                    val = None
            if isinstance(val, list) and len(val) == 2:
                visible_parts += 1
        V_raw = visible_parts / len(upper_parts)

        # emotion
        E_raw = row.get('emotion_similarity_to_profile', np.nan)

        # combine scores and store
        results.append({
            'image_path': row.get('image_path'),
            'subject_politician': shorthand,
            'C_raw': C_raw,
            'S_raw': S_raw,
            'V_raw': V_raw,
            'E_raw': E_raw
        })

    score_df = pd.DataFrame(results)

    # Normalize scores
    for col in ['C_raw', 'S_raw', 'V_raw', 'E_raw']:
        score_df[col[0]] = normalize(score_df[col])

    # Calculate Iconicity score
    score_df['Iconicity'] = 0.25 * score_df['C'] + 0.25 * score_df['S'] + 0.25 * score_df['V'] + 0.25 * score_df['E']

    return score_df


In [ ]:
# Run iconicity score calculator
iconicity_score_df = compute_iconicity(df)
iconicity_score_df = iconicity_score_df[['image_path', 'C', 'S', 'V', 'E', 'Iconicity']]

# Z-Standardize iconicity score
scaler = StandardScaler()
csve_z = scaler.fit_transform(iconicity_score_df[['C', 'S', 'V', 'E']])
iconicity_score_df[['C_z', 'S_z', 'V_z', 'E_z']] = csve_z
iconicity_score_df['Iconicity_z'] = csve_z.mean(axis=1)

# use min max for iconcitiy score
iconicity_score_df['Iconicity_z'] = MinMaxScaler().fit_transform(iconicity_score_df[['Iconicity_z']])

# Merge into df
df = df.merge(iconicity_score_df, on='image_path', how='left')


In [ ]:
# df.to_excel('/content/drive/MyDrive/Thesis/analysis/df_final_analysis2.xlsx')